# Sliced Normal: fast fitting demo

Complete workflow: load sample data, initialize with FMLE, fit the cached default,
check approximate likelihood and final stationarity, sample both models, and plot.

Use Python 3.11. First follow the environment commands in the root README and
select **Python (Sliced Normal Fast)** as the kernel. If needed, the optional
installation cell below installs into your currently selected kernel; restart
that kernel afterward. No repository cloning or deletion is performed.

MCMC retains the original unbounded support while normalization uses a box.
These plots are exploratory; chain convergence is not established by this demo.

In [ ]:
# Optional: from the repository root, install once, then restart this kernel.
# %pip install -e ".[demo]"


In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'sliced_normals/optimization.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or Demo folder.')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'Demo'))
from workflow import fit_demo, sample_and_plot
from IPython.display import display, Image
print('Python:', sys.executable)
print('Repository:', ROOT)


In [ ]:
# Increase budgets here for your own convergence study.
ITERATIONS = 2000
GRID = 5000
EVALUATION_GRID = 50000
STEPS = 2000
BURNIN = 300
THIN = 5
OUTPUT = ROOT/'Demo/outputs'


## FMLE, fast fitting, and independent normalization
The grid used for likelihood evaluation is independent of fitting and shared between the two estimates. This is in-sample mean likelihood, not held-out performance. The endpoint gradient is recomputed; a false tolerance flag means the requested final stationarity has not been verified.

In [ ]:
data, B_fmle, B, metrics = fit_demo(ITERATIONS, GRID, EVALUATION_GRID)
display(data.head())
print(json.dumps(metrics, indent=2))


## Sampling and pairplots
`STEPS` is steps per walker, not the final retained row count. Inspect mixing, effective sample size, and mode coverage in subsequent experiments. Sampling and feature order match the original implementation.

In [ ]:
frames = sample_and_plot(data, B_fmle, B, OUTPUT, STEPS, BURNIN, THIN)
for name, frame in frames.items():
    print(name, 'rows:', len(frame))
    display(Image(filename=str(OUTPUT/(name+'_pairplot.png'))))
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'metrics.json').write_text(json.dumps(metrics, indent=2)+'\n')
import numpy as np
np.save(OUTPUT/'B_fast.npy', B)
np.save(OUTPUT/'B_fmle.npy', B_fmle)
print('Saved matrices, metrics, sample CSVs, and plots to:', OUTPUT)
